# 📓 Lab 02: Agentic ReAct & Human-in-the-Loop Breakpoint Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TontonYuta/Applied-AI-Engineering-Handbook/blob/main/notebooks/02_agentic_langgraph_hitl.ipynb)

> **Mục tiêu bài lab:**
> 1. Triển khai chu trình ReAct chuẩn mực (Thought -> Action -> Observation).
> 2. Thiết lập cơ chế đăng ký và thực thi công cụ (Tool Calling Registry).
> 3. Tích hợp điểm ngắt an toàn `state.interrupt()` (Human-in-the-Loop) cho giao dịch lớn.

In [ ]:
# Cài đặt thư viện cần thiết
!pip install -q pydantic rich

In [ ]:
import re
import uuid
import time
from typing import Dict, Any, Callable, Optional

print("✅ Môi trường Agentic sẵn sàng!")

## Bước 1: Khởi tạo Tool Registry & Các Công Cụ Doanh Nghiệp

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}
    def register(self, name: str, desc: str, fn: Callable):
        self.tools[name] = {"desc": desc, "fn": fn}
    def execute(self, name: str, **kwargs):
        if name not in self.tools: return f"Tool {name} not found"
        return self.tools[name]["fn"](**kwargs)

registry = ToolRegistry()
registry.register("get_order", "Tra cứu đơn hàng", lambda order_id: {"id": order_id, "amount": 25_000_000, "status": "DELIVERED"})
registry.register("check_policy", "Kiểm tra chính sách hoàn tiền", lambda order_id: "Được phép hoàn tiền trong 7 ngày")

print("✔ Đã đăng ký các công cụ doanh nghiệp!")

## Bước 2: Thiết lập Cổng Phê Duyệt An Toàn (Human-in-the-Loop)

In [ ]:
class HITLGate:
    def __init__(self, limit: float = 5_000_000.0):
        self.limit = limit
        self.pending = {}
    def process_refund(self, order_id: str, amount: float):
        if amount <= self.limit:
            return f"[AUTO APPROVED] Đã tự động giải ngân {amount:,.0f} VNĐ cho đơn {order_id}"
        ticket_id = f"TICKET-{uuid.uuid4().hex[:6].upper()}"
        self.pending[ticket_id] = {"order_id": order_id, "amount": amount}
        return f"[INTERRUPT BREAKPOINT] Số tiền {amount:,.0f} VNĐ vượt hạn mức. Ticket {ticket_id} chờ Quản lý duyệt!"

gate = HITLGate(limit=5_000_000.0)
print(gate.process_refund("ORD-SMALL", 1_500_000))
print(gate.process_refund("ORD-LARGE", 25_000_000))